In [1]:
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image as PILImage
from google.colab import drive


# =============================================================================
# SETSTOP-KGQA NOTEBOOK 10
# Final paper analysis, statistical tests, tables and figures
#
# Runtime: CPU
# Model inference: No
# Expected runtime: approximately 1–3 minutes
# =============================================================================

START_TIME = time.time()

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
DATA_DIR = ROOT / "data/extracted/webqsp"
OUTPUT_ROOT = ROOT / "outputs/final_paper_analysis_v1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

END_TO_END_PER_QUESTION = (
    ROOT
    / "outputs/end_to_end_test_v1/"
    / "end_to_end_per_question_v1.jsonl"
)

END_TO_END_SUMMARY = (
    ROOT
    / "outputs/end_to_end_test_v1/"
    / "end_to_end_summary_v1.json"
)

COST_PER_QUESTION = (
    ROOT
    / "outputs/cost_matched_ablation_v1/"
    / "test_budget_per_question_v1.jsonl"
)

COST_RESULTS = (
    ROOT
    / "outputs/cost_matched_ablation_v1/"
    / "test_budget_results_v1.csv"
)

COST_COMPARISONS = (
    ROOT
    / "outputs/cost_matched_ablation_v1/"
    / "cost_matched_comparisons_v1.json"
)

COST_PAIRED = (
    ROOT
    / "outputs/cost_matched_ablation_v1/"
    / "paired_bootstrap_cost_matched_v1.json"
)

COST_SUMMARY = (
    ROOT
    / "outputs/cost_matched_ablation_v1/"
    / "cost_matched_ablation_summary_v1.json"
)


# =============================================================================
# 1. GENERAL HELPERS
# =============================================================================

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path):
    rows = []

    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as error:
                raise ValueError(
                    f"Invalid JSON at {path}:{line_number}"
                ) from error

    return rows


def write_json(path, value):
    path.write_text(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def find_result_file(
    filename,
    preferred_directory=None,
):
    matches = sorted(ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {ROOT}"
        )

    if preferred_directory is not None:
        preferred = [
            path
            for path in matches
            if preferred_directory in str(path.parent)
        ]

        if preferred:
            return preferred[0]

    if len(matches) > 1:
        hashes = {
            sha256_file(path)
            for path in matches
        }

        if len(hashes) == 1:
            print(
                f"Found identical copies of {filename}; "
                f"using {matches[0]}"
            )
            return matches[0]

        print(
            f"WARNING: multiple non-identical copies "
            f"of {filename} were found."
        )
        print("Using:", matches[0])

    return matches[0]


def safe_float(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None

    if not np.isfinite(number):
        return None

    return number


def fmt(value, digits=3):
    number = safe_float(value)

    if number is None:
        return "--"

    return f"{number:.{digits}f}"


def latex_escape(text):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    return "".join(
        replacements.get(character, character)
        for character in str(text)
    )


def exact_two_sided_mcnemar_pvalue(
    first_only,
    second_only,
):
    discordant = (
        int(first_only)
        + int(second_only)
    )

    if discordant == 0:
        return 1.0

    smaller = min(
        int(first_only),
        int(second_only),
    )

    lower_tail = (
        sum(
            math.comb(discordant, k)
            for k in range(smaller + 1)
        )
        / (2 ** discordant)
    )

    return float(
        min(1.0, 2.0 * lower_tail)
    )


def png_to_pdf(
    png_path,
    pdf_path,
):
    """
    Use Pillow for PDF output to avoid the Matplotlib
    backend_pdf/FontPath error encountered previously.
    """

    with PILImage.open(png_path) as source:
        if source.mode == "RGBA":
            background = PILImage.new(
                "RGB",
                source.size,
                "white",
            )

            background.paste(
                source,
                mask=source.getchannel("A"),
            )

            image = background
        else:
            image = source.convert("RGB")

        image.save(
            pdf_path,
            format="PDF",
            resolution=300.0,
        )

    assert (
        pdf_path.exists()
        and pdf_path.stat().st_size > 0
    )


# =============================================================================
# 2. LOCATE RESULTS FROM EARLIER NOTEBOOKS
# =============================================================================

EXTERNAL_TABLE = find_result_file(
    "external_baseline_results_table_v1.csv"
)

EXTERNAL_SUMMARY = find_result_file(
    "external_baseline_summary_v1.json"
)

EXTERNAL_MANIFEST = find_result_file(
    "external_baseline_run_manifest_v1.json"
)

EXTERNAL_PAIRED = find_result_file(
    "paired_bootstrap_external_v1.json",
    preferred_directory=(
        "operating_point_and_cost_v1"
    ),
)

OPERATING_SUMMARY = find_result_file(
    "operating_point_and_cost_summary_v1.json",
    preferred_directory=(
        "operating_point_and_cost_v1"
    ),
)

required_paths = [
    DATA_DIR / "test_simple.json",
    END_TO_END_PER_QUESTION,
    END_TO_END_SUMMARY,
    COST_PER_QUESTION,
    COST_RESULTS,
    COST_COMPARISONS,
    COST_PAIRED,
    COST_SUMMARY,
    EXTERNAL_TABLE,
    EXTERNAL_SUMMARY,
    EXTERNAL_MANIFEST,
    EXTERNAL_PAIRED,
    OPERATING_SUMMARY,
]

for path in required_paths:
    assert path.exists(), (
        f"Missing required input file: {path}"
    )


print("=" * 110)
print(
    "SETSTOP-KGQA FINAL PAPER ANALYSIS "
    "AND LATEX TABLES"
)
print("=" * 110)
print(
    "Runtime: CPU — no GPU or model "
    "inference is required"
)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Output directory:", OUTPUT_ROOT)
print("=" * 110)


# =============================================================================
# 3. LOAD AND VALIDATE ALL RESULTS
# =============================================================================

e2e_rows = read_jsonl(
    END_TO_END_PER_QUESTION
)

cost_rows = read_jsonl(
    COST_PER_QUESTION
)

e2e_summary = read_json(
    END_TO_END_SUMMARY
)

cost_comparisons = read_json(
    COST_COMPARISONS
)

cost_paired = read_json(
    COST_PAIRED
)

cost_summary = read_json(
    COST_SUMMARY
)

external_summary = read_json(
    EXTERNAL_SUMMARY
)

external_manifest = read_json(
    EXTERNAL_MANIFEST
)

external_paired = read_json(
    EXTERNAL_PAIRED
)

operating_summary = read_json(
    OPERATING_SUMMARY
)

budget_df = pd.read_csv(
    COST_RESULTS
)

external_df = pd.read_csv(
    EXTERNAL_TABLE
)

e2e_df = pd.DataFrame(
    e2e_rows
)

cost_per_question_df = pd.DataFrame(
    cost_rows
)

assert len(e2e_df) == 5152, (
    f"Expected 5152 end-to-end rows, "
    f"found {len(e2e_df)}"
)

assert len(cost_per_question_df) == 736 * 18, (
    f"Expected {736 * 18} cost rows, "
    f"found {len(cost_per_question_df)}"
)

assert len(budget_df) == 18, (
    f"Expected 18 budget configurations, "
    f"found {len(budget_df)}"
)

assert (
    external_manifest["test_questions"]
    == 736
)

assert (
    cost_summary["regression_check"]["status"]
    == "PASS"
)

assert (
    cost_summary["regression_check"]
    ["maximum_absolute_difference"]
    == 0.0
)

test_qids = sorted(
    cost_per_question_df.loc[
        (
            cost_per_question_df["policy"]
            == "setstop"
        )
        & (
            cost_per_question_df["budget_cap"]
            == 12
        ),
        "qid",
    ]
    .astype(str)
    .unique()
    .tolist()
)

assert len(test_qids) == 736

e2e_qids = set(
    e2e_df.loc[
        e2e_df["policy"] == "setstop_e2e",
        "qid",
    ].astype(str)
)

assert set(test_qids) == e2e_qids, (
    "The cost-ablation and end-to-end files "
    "do not contain the same test questions."
)

qid_digest = (
    external_manifest["test_qid_sha256"]
)

assert (
    qid_digest
    == operating_summary["test_qid_sha256"]
)

print(
    "Controlled test subset verified: "
    "736 questions"
)
print("QID SHA-256:", qid_digest)
print(
    "Notebook-06 cap-12 reproduction: PASS"
)


# =============================================================================
# 4. LOAD QUESTION TEXT AND GOLD-ANSWER LABELS
# =============================================================================

def get_question_id(
    example,
    line_number,
):
    for key in [
        "id",
        "qid",
        "question_id",
    ]:
        if key in example:
            return str(example[key])

    question = str(
        example.get(
            "question",
            example.get("text", ""),
        )
    ).strip()

    digest = hashlib.sha1(
        question.encode("utf-8")
    ).hexdigest()[:12]

    return (
        f"test_{line_number}_{digest}"
    )


def answer_label(answer):
    if isinstance(answer, dict):
        for key in [
            "text",
            "name",
            "label",
            "kb_id",
            "id",
        ]:
            if answer.get(key) is not None:
                return str(answer[key])

    return str(answer)


question_metadata = {}

with open(
    DATA_DIR / "test_simple.json",
    encoding="utf-8",
) as file:

    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        example = json.loads(line)

        qid = get_question_id(
            example,
            line_number,
        )

        subgraph = example.get(
            "subgraph",
            {},
        )

        graph_edges = subgraph.get(
            "tuples",
            subgraph.get("triples", []),
        )

        graph_entities = subgraph.get(
            "entities",
            [],
        )

        answers = example.get(
            "answers",
            [],
        )

        question_metadata[qid] = {
            "question": str(
                example.get(
                    "question",
                    example.get(
                        "text",
                        example.get("query", ""),
                    ),
                )
            ).strip(),
            "gold_answer_labels": [
                answer_label(answer)
                for answer in answers
            ],
            "raw_gold_answer_count": len(
                answers
            ),
            "topic_entity_count": len(
                example.get("entities", [])
            ),
            "input_graph_entities": len(
                graph_entities
            ),
            "input_graph_edges": len(
                graph_edges
            ),
        }


missing_metadata = sorted(
    set(test_qids)
    - set(question_metadata)
)

assert not missing_metadata, (
    "Question metadata is missing for: "
    f"{missing_metadata[:5]}"
)


# =============================================================================
# 5. FINAL MAIN COMPARISON TABLE
# =============================================================================

external_name_map = {
    "Relation heuristic, fixed budget":
        "Relation fixed (cap 12)",
    "Learned ranker, fixed budget":
        "Learned ranker (cap 12)",
    "SetStop":
        "SetStop",
    "Oracle ranker":
        "Oracle ranker",
    "NSM+h":
        "NSM+h",
    "ReaRev":
        "ReaRev",
}

setstop_edges = float(
    operating_summary[
        "setstop"
    ][
        "mean_input_edges_exposed"
    ]
)

full_graph_edges = (
    setstop_edges
    + float(
        external_paired[
            "comparisons"
        ][
            "NSM+h"
        ][
            "input_edges_saved"
        ][
            "mean_difference"
        ]
    )
)

main_rows = []

for _, row in external_df.iterrows():
    original_name = str(
        row["Method"]
    )

    if original_name not in external_name_map:
        continue

    method = external_name_map[
        original_name
    ]

    is_external = (
        original_name
        in {"NSM+h", "ReaRev"}
    )

    actions = (
        None
        if is_external
        else safe_float(row.get("Actions"))
    )

    input_edges = (
        full_graph_edges
        if is_external
        else safe_float(
            row.get("Retrieved Edges")
        )
    )

    main_rows.append({
        "group": (
            "Full-graph baseline"
            if is_external
            else "Sequential search"
        ),
        "method": method,
        "macro_precision": float(
            row["Macro Precision"]
        ),
        "macro_recall": float(
            row["Macro Recall"]
        ),
        "macro_f1": float(
            row["Macro F1"]
        ),
        "micro_f1": float(
            row["Micro F1"]
        ),
        "exact_match": float(
            row["Exact Match"]
        ),
        "complete_recall": float(
            row["Complete Recall"]
        ),
        "mean_actions": actions,
        "mean_input_edges": input_edges,
    })


main_results_df = pd.DataFrame(
    main_rows
)

assert len(main_results_df) == 6

main_results_csv = (
    OUTPUT_ROOT
    / "main_results_table_v1.csv"
)

main_results_df.to_csv(
    main_results_csv,
    index=False,
)


def create_main_latex_table(dataframe):
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4.5pt}",
        r"\begin{tabular}{llrrrrrrrr}",
        r"\toprule",
        (
            r"Setting & Method & P & R & F1 "
            r"& Micro-F1 & EM & CR & Actions "
            r"& Edges \\"
        ),
        r"\midrule",
    ]

    groups = [
        "Sequential search",
        "Full-graph baseline",
    ]

    for group_index, group in enumerate(
        groups
    ):
        group_rows = dataframe[
            dataframe["group"] == group
        ]

        for row_index, (_, row) in enumerate(
            group_rows.iterrows()
        ):
            setting = (
                latex_escape(group)
                if row_index == 0
                else ""
            )

            method = latex_escape(
                row["method"]
            )

            if row["method"] == "SetStop":
                method = r"\textbf{SetStop}"

            lines.append(
                f"{setting} & {method} & "
                f"{fmt(row['macro_precision'])} & "
                f"{fmt(row['macro_recall'])} & "
                f"{fmt(row['macro_f1'])} & "
                f"{fmt(row['micro_f1'])} & "
                f"{fmt(row['exact_match'])} & "
                f"{fmt(row['complete_recall'])} & "
                f"{fmt(row['mean_actions'], 2)} & "
                f"{fmt(row['mean_input_edges'], 1)} "
                + r"\\"
            )

        if group_index < len(groups) - 1:
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        (
            r"\caption{Answer-set results on the "
            r"controlled 736-question WebQSP "
            r"multi-answer subset. P, R, EM, and CR "
            r"denote macro precision, macro recall, "
            r"exact match, and complete-answer recall. "
            r"Edges denotes mean input triples exposed, "
            r"a structural graph-access proxy rather "
            r"than FLOPs. Full-graph baselines consume "
            r"the complete question-specific graph and "
            r"do not take sequential actions.}"
        ),
        r"\label{tab:main_results}",
        r"\end{table*}",
    ])

    return "\n".join(lines) + "\n"


main_results_tex = (
    OUTPUT_ROOT
    / "main_results_table_v1.tex"
)

main_results_tex.write_text(
    create_main_latex_table(
        main_results_df
    ),
    encoding="utf-8",
)


# =============================================================================
# 6. COST-MATCHED COMPARISON
# =============================================================================

def get_budget_row(
    policy,
    cap,
):
    matches = budget_df[
        (
            budget_df["policy"]
            == policy
        )
        & (
            budget_df["budget_cap"]
            == cap
        )
    ]

    assert len(matches) == 1, (
        f"Expected one row for "
        f"{policy}, cap={cap}; "
        f"found {len(matches)}"
    )

    return matches.iloc[0]


cost_configurations = [
    (
        "relation_fixed",
        6,
        "Relation fixed (cap 6)",
    ),
    (
        "learned_ranker_fixed",
        6,
        "Learned ranker (cap 6)",
    ),
    (
        "setstop",
        12,
        "SetStop (adaptive, cap 12)",
    ),
]

cost_table_rows = []

for policy, cap, label in cost_configurations:
    row = get_budget_row(
        policy,
        cap,
    )

    cost_table_rows.append({
        "method": label,
        "macro_precision": float(
            row["macro_precision"]
        ),
        "macro_recall": float(
            row["macro_recall"]
        ),
        "macro_f1": float(
            row["macro_f1"]
        ),
        "f1_ci95_lower": float(
            row["macro_f1_ci95_lower"]
        ),
        "f1_ci95_upper": float(
            row["macro_f1_ci95_upper"]
        ),
        "exact_match": float(
            row["exact_match"]
        ),
        "complete_recall": float(
            row["complete_recall"]
        ),
        "mean_actions": float(
            row["mean_actions"]
        ),
        "mean_retrieved_edges": float(
            row["mean_retrieved_edges"]
        ),
    })


cost_table_df = pd.DataFrame(
    cost_table_rows
)

cost_table_csv = (
    OUTPUT_ROOT
    / "cost_matched_table_v1.csv"
)

cost_table_df.to_csv(
    cost_table_csv,
    index=False,
)


primary_pair = (
    cost_paired[
        "test"
    ][
        "primary_cost_matched_vs_learned"
    ]
)

delta_f1 = (
    primary_pair[
        "metrics"
    ][
        "answer_f1"
    ]
)

delta_exact = (
    primary_pair[
        "metrics"
    ][
        "exact_match"
    ]
)

delta_complete = (
    primary_pair[
        "metrics"
    ][
        "complete_recall"
    ]
)

delta_actions = (
    primary_pair[
        "metrics"
    ][
        "actions"
    ]
)

delta_edges = (
    primary_pair[
        "metrics"
    ][
        "retrieved_edges"
    ]
)


def create_cost_latex_table(dataframe):
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{lrrrrrrr}",
        r"\toprule",
        (
            r"Method & P & R & F1 & EM & CR "
            r"& Actions & Edges \\"
        ),
        r"\midrule",
    ]

    for _, row in dataframe.iterrows():
        method = latex_escape(
            row["method"]
        )

        if row["method"].startswith(
            "SetStop"
        ):
            method = (
                r"\textbf{SetStop "
                r"(adaptive, cap 12)}"
            )

        lines.append(
            f"{method} & "
            f"{fmt(row['macro_precision'])} & "
            f"{fmt(row['macro_recall'])} & "
            f"{fmt(row['macro_f1'])} & "
            f"{fmt(row['exact_match'])} & "
            f"{fmt(row['complete_recall'])} & "
            f"{fmt(row['mean_actions'], 2)} & "
            f"{fmt(row['mean_retrieved_edges'], 1)} "
            + r"\\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        (
            r"\caption{Action-matched comparison. "
            r"SetStop uses 5.95 actions on average, "
            r"while fixed policies use six. Against "
            r"the learned fixed-budget ranker, SetStop "
            f"improves macro F1 by "
            f"{delta_f1['mean_difference_first_minus_second']:.3f} "
            f"(paired 95\\% CI "
            f"[{delta_f1['ci95_lower']:.3f}, "
            f"{delta_f1['ci95_upper']:.3f}]). "
            r"The differences in complete recall and "
            r"exposed edges are not statistically "
            r"significant.}"
        ),
        r"\label{tab:cost_matched}",
        r"\end{table*}",
    ])

    return "\n".join(lines) + "\n"


cost_table_tex = (
    OUTPUT_ROOT
    / "cost_matched_table_v1.tex"
)

cost_table_tex.write_text(
    create_cost_latex_table(
        cost_table_df
    ),
    encoding="utf-8",
)


# =============================================================================
# 7. COMPLETE ACTION-BUDGET ABLATION TABLE
# =============================================================================

budget_order = [
    2,
    4,
    6,
    8,
    10,
    12,
]

budget_table_rows = []

for cap in budget_order:
    relation_row = get_budget_row(
        "relation_fixed",
        cap,
    )

    learned_row = get_budget_row(
        "learned_ranker_fixed",
        cap,
    )

    setstop_row_at_cap = get_budget_row(
        "setstop",
        cap,
    )

    same_cap_comparison = (
        cost_paired[
            "test"
        ][
            "same_cap"
        ][
            str(cap)
        ][
            "versus_learned"
        ]
    )

    same_cap_f1 = (
        same_cap_comparison[
            "metrics"
        ][
            "answer_f1"
        ]
    )

    budget_table_rows.append({
        "budget_cap": cap,
        "relation_f1": float(
            relation_row["macro_f1"]
        ),
        "learned_f1": float(
            learned_row["macro_f1"]
        ),
        "setstop_f1": float(
            setstop_row_at_cap["macro_f1"]
        ),
        "setstop_exact": float(
            setstop_row_at_cap["exact_match"]
        ),
        "setstop_complete": float(
            setstop_row_at_cap[
                "complete_recall"
            ]
        ),
        "setstop_actions": float(
            setstop_row_at_cap[
                "mean_actions"
            ]
        ),
        "setstop_edges": float(
            setstop_row_at_cap[
                "mean_retrieved_edges"
            ]
        ),
        "setstop_minus_learned_f1": float(
            same_cap_f1[
                "mean_difference_first_minus_second"
            ]
        ),
        "f1_difference_significant": bool(
            same_cap_f1[
                "significant_at_0_05"
            ]
        ),
    })


budget_table_df = pd.DataFrame(
    budget_table_rows
)

budget_table_csv = (
    OUTPUT_ROOT
    / "budget_ablation_table_v1.csv"
)

budget_table_df.to_csv(
    budget_table_csv,
    index=False,
)


budget_latex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{rrrrrrrrr}",
    r"\toprule",
    (
        r"Cap & Relation F1 & Learned F1 "
        r"& SetStop F1 & $\Delta$F1 & EM "
        r"& CR & Actions & Edges \\"
    ),
    r"\midrule",
]

for _, row in budget_table_df.iterrows():
    significance_marker = (
        r"$^{*}$"
        if row["f1_difference_significant"]
        else ""
    )

    budget_latex_lines.append(
        f"{int(row['budget_cap'])} & "
        f"{fmt(row['relation_f1'])} & "
        f"{fmt(row['learned_f1'])} & "
        f"{fmt(row['setstop_f1'])} & "
        f"{fmt(row['setstop_minus_learned_f1'])}"
        f"{significance_marker} & "
        f"{fmt(row['setstop_exact'])} & "
        f"{fmt(row['setstop_complete'])} & "
        f"{fmt(row['setstop_actions'], 2)} & "
        f"{fmt(row['setstop_edges'], 1)} "
        + r"\\"
    )


budget_latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    (
        r"\caption{Test performance across the "
        r"predefined maximum action budgets. "
        r"$\Delta$F1 is SetStop minus the learned "
        r"fixed-budget ranker at the same cap; "
        r"$^{*}$ indicates a paired bootstrap "
        r"95\% confidence interval excluding zero.}"
    ),
    r"\label{tab:budget_ablation}",
    r"\end{table*}",
])


budget_table_tex = (
    OUTPUT_ROOT
    / "budget_ablation_table_v1.tex"
)

budget_table_tex.write_text(
    "\n".join(budget_latex_lines) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 8. PER-QUESTION DATA FOR ERROR ANALYSIS
# =============================================================================

setstop_df = cost_per_question_df[
    (
        cost_per_question_df["policy"]
        == "setstop"
    )
    & (
        cost_per_question_df["budget_cap"]
        == 12
    )
].copy()

learned6_df = cost_per_question_df[
    (
        cost_per_question_df["policy"]
        == "learned_ranker_fixed"
    )
    & (
        cost_per_question_df["budget_cap"]
        == 6
    )
].copy()

relation6_df = cost_per_question_df[
    (
        cost_per_question_df["policy"]
        == "relation_fixed"
    )
    & (
        cost_per_question_df["budget_cap"]
        == 6
    )
].copy()

assert (
    len(setstop_df)
    == len(learned6_df)
    == len(relation6_df)
    == 736
)

assert (
    set(setstop_df["qid"])
    == set(learned6_df["qid"])
    == set(relation6_df["qid"])
)

for key in [
    "question",
    "gold_answer_labels",
    "input_graph_entities",
    "input_graph_edges",
]:
    setstop_df[key] = setstop_df[
        "qid"
    ].map(
        lambda qid: question_metadata[
            str(qid)
        ][key]
    )


def assign_error_category(row):
    if int(row["exact_match"]) == 1:
        return "Exact"

    if int(
        row["predicted_answer_count"]
    ) == 0:
        return "Empty prediction"

    false_positives = int(
        row["false_positive_count"]
    )

    false_negatives = int(
        row["false_negative_count"]
    )

    if (
        false_positives == 0
        and false_negatives > 0
    ):
        return "Incomplete only"

    if (
        false_positives > 0
        and false_negatives == 0
    ):
        return "Over-prediction only"

    return "Mixed FP and FN"


def assign_stop_outcome(row):
    reason = str(
        row["stop_reason"]
    )

    if reason == "action_budget":
        return "Budget exhausted"

    if reason == "empty_frontier":
        return "Frontier exhausted"

    if (
        reason == "learned_stop"
        and int(row["premature_stop"]) == 1
    ):
        return "Premature learned stop"

    if reason == "learned_stop":
        return "Safe learned stop"

    return f"Other: {reason}"


def answer_count_group(count):
    count = int(count)

    if count == 2:
        return "2 answers"

    if count <= 5:
        return "3–5 answers"

    return "6+ answers"


setstop_df["error_category"] = (
    setstop_df.apply(
        assign_error_category,
        axis=1,
    )
)

setstop_df["stop_outcome"] = (
    setstop_df.apply(
        assign_stop_outcome,
        axis=1,
    )
)

setstop_df["answer_count_group"] = (
    setstop_df[
        "gold_answer_count"
    ].map(answer_count_group)
)


def summarize_groups(
    dataframe,
    group_column,
    ordered_groups,
):
    rows = []

    for group in ordered_groups:
        subset = dataframe[
            dataframe[group_column] == group
        ]

        if subset.empty:
            continue

        rows.append({
            group_column: group,
            "questions": int(len(subset)),
            "percentage": float(
                100.0
                * len(subset)
                / len(dataframe)
            ),
            "macro_precision": float(
                subset["precision"].mean()
            ),
            "macro_recall": float(
                subset["recall"].mean()
            ),
            "macro_f1": float(
                subset["answer_f1"].mean()
            ),
            "exact_match": float(
                subset["exact_match"].mean()
            ),
            "complete_recall": float(
                subset["complete_recall"].mean()
            ),
            "mean_actions": float(
                subset["actions"].mean()
            ),
            "mean_edges": float(
                subset[
                    "retrieved_edges"
                ].mean()
            ),
            "mean_false_positives": float(
                subset[
                    "false_positive_count"
                ].mean()
            ),
            "mean_false_negatives": float(
                subset[
                    "false_negative_count"
                ].mean()
            ),
        })

    return pd.DataFrame(rows)


error_order = [
    "Exact",
    "Incomplete only",
    "Over-prediction only",
    "Mixed FP and FN",
    "Empty prediction",
]

stop_order = [
    "Safe learned stop",
    "Premature learned stop",
    "Frontier exhausted",
    "Budget exhausted",
]

stop_order += sorted(
    group
    for group in set(
        setstop_df["stop_outcome"]
    )
    if group not in stop_order
)

answer_order = [
    "2 answers",
    "3–5 answers",
    "6+ answers",
]

error_df = summarize_groups(
    setstop_df,
    "error_category",
    error_order,
)

stop_df = summarize_groups(
    setstop_df,
    "stop_outcome",
    stop_order,
)

answer_df = summarize_groups(
    setstop_df,
    "answer_count_group",
    answer_order,
)

assert int(
    error_df["questions"].sum()
) == 736

assert int(
    stop_df["questions"].sum()
) == 736

assert int(
    answer_df["questions"].sum()
) == 736


error_csv = (
    OUTPUT_ROOT
    / "error_analysis_table_v1.csv"
)

stop_csv = (
    OUTPUT_ROOT
    / "stop_outcome_analysis_v1.csv"
)

answer_csv = (
    OUTPUT_ROOT
    / "answer_count_analysis_v1.csv"
)

error_df.to_csv(
    error_csv,
    index=False,
)

stop_df.to_csv(
    stop_csv,
    index=False,
)

answer_df.to_csv(
    answer_csv,
    index=False,
)


def write_group_latex_table(
    dataframe,
    label_column,
    caption,
    table_label,
    output_path,
):
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{lrrrrrrrr}",
        r"\toprule",
        (
            r"Group & N & \% & P & R & F1 "
            r"& EM & CR & Actions \\"
        ),
        r"\midrule",
    ]

    for _, row in dataframe.iterrows():
        lines.append(
            f"{latex_escape(row[label_column])} & "
            f"{int(row['questions'])} & "
            f"{row['percentage']:.1f} & "
            f"{fmt(row['macro_precision'])} & "
            f"{fmt(row['macro_recall'])} & "
            f"{fmt(row['macro_f1'])} & "
            f"{fmt(row['exact_match'])} & "
            f"{fmt(row['complete_recall'])} & "
            f"{fmt(row['mean_actions'], 2)} "
            + r"\\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{table_label}}}",
        r"\end{table*}",
    ])

    output_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )


error_tex = (
    OUTPUT_ROOT
    / "error_analysis_table_v1.tex"
)

stop_tex = (
    OUTPUT_ROOT
    / "stop_outcome_analysis_v1.tex"
)

answer_tex = (
    OUTPUT_ROOT
    / "answer_count_analysis_v1.tex"
)

write_group_latex_table(
    dataframe=error_df,
    label_column="error_category",
    caption=(
        "SetStop error categories on the "
        "WebQSP multi-answer test subset."
    ),
    table_label="tab:error_analysis",
    output_path=error_tex,
)

write_group_latex_table(
    dataframe=stop_df,
    label_column="stop_outcome",
    caption=(
        "SetStop performance according to "
        "the stopping outcome."
    ),
    table_label="tab:stop_outcomes",
    output_path=stop_tex,
)

write_group_latex_table(
    dataframe=answer_df,
    label_column="answer_count_group",
    caption=(
        "SetStop performance by gold "
        "answer-set cardinality."
    ),
    table_label="tab:answer_count",
    output_path=answer_tex,
)


# =============================================================================
# 9. PAIRED QUESTION-LEVEL ANALYSIS
# =============================================================================

comparison_df = setstop_df.merge(
    learned6_df[
        [
            "qid",
            "answer_f1",
            "exact_match",
            "complete_recall",
            "actions",
            "retrieved_edges",
            "precision",
            "recall",
        ]
    ],
    on="qid",
    how="inner",
    suffixes=(
        "_setstop",
        "_learned6",
    ),
)

assert len(comparison_df) == 736

tolerance = 1e-12

f1_difference = (
    comparison_df[
        "answer_f1_setstop"
    ]
    - comparison_df[
        "answer_f1_learned6"
    ]
)

f1_wins = int(
    (
        f1_difference > tolerance
    ).sum()
)

f1_losses = int(
    (
        f1_difference < -tolerance
    ).sum()
)

f1_ties = int(
    len(comparison_df)
    - f1_wins
    - f1_losses
)

exact_setstop_only = int(
    (
        (
            comparison_df[
                "exact_match_setstop"
            ] == 1
        )
        & (
            comparison_df[
                "exact_match_learned6"
            ] == 0
        )
    ).sum()
)

exact_learned_only = int(
    (
        (
            comparison_df[
                "exact_match_setstop"
            ] == 0
        )
        & (
            comparison_df[
                "exact_match_learned6"
            ] == 1
        )
    ).sum()
)

complete_setstop_only = int(
    (
        (
            comparison_df[
                "complete_recall_setstop"
            ] == 1
        )
        & (
            comparison_df[
                "complete_recall_learned6"
            ] == 0
        )
    ).sum()
)

complete_learned_only = int(
    (
        (
            comparison_df[
                "complete_recall_setstop"
            ] == 0
        )
        & (
            comparison_df[
                "complete_recall_learned6"
            ] == 1
        )
    ).sum()
)


paired_diagnostics = {
    "comparison": (
        "SetStop adaptive cap 12 versus "
        "learned fixed cap 6"
    ),
    "questions": 736,
    "f1_win_tie_loss": {
        "setstop_wins": f1_wins,
        "ties": f1_ties,
        "setstop_losses": f1_losses,
    },
    "exact_match_mcnemar": {
        "setstop_only_correct":
            exact_setstop_only,
        "learned_only_correct":
            exact_learned_only,
        "exact_two_sided_p_value":
            exact_two_sided_mcnemar_pvalue(
                exact_setstop_only,
                exact_learned_only,
            ),
    },
    "complete_recall_mcnemar": {
        "setstop_only_complete":
            complete_setstop_only,
        "learned_only_complete":
            complete_learned_only,
        "exact_two_sided_p_value":
            exact_two_sided_mcnemar_pvalue(
                complete_setstop_only,
                complete_learned_only,
            ),
    },
    "paired_bootstrap":
        primary_pair,
}

paired_diagnostics_path = (
    OUTPUT_ROOT
    / "paired_diagnostics_v1.json"
)

write_json(
    paired_diagnostics_path,
    paired_diagnostics,
)


# =============================================================================
# 10. SELECT QUALITATIVE EXAMPLES
# =============================================================================

diagnostic_df = setstop_df.merge(
    learned6_df[
        [
            "qid",
            "answer_f1",
            "exact_match",
            "complete_recall",
            "predicted_answer_count",
            "true_positive_count",
            "false_positive_count",
            "false_negative_count",
            "actions",
            "retrieved_edges",
        ]
    ],
    on="qid",
    suffixes=(
        "_setstop",
        "_learned6",
    ),
)

diagnostic_df["f1_improvement"] = (
    diagnostic_df[
        "answer_f1_setstop"
    ]
    - diagnostic_df[
        "answer_f1_learned6"
    ]
)

selected_examples = []
used_qids = set()


def select_examples(
    category,
    candidates,
    count=2,
    ascending=False,
):
    available = candidates[
        ~candidates[
            "qid"
        ].astype(str).isin(used_qids)
    ].copy()

    if "selection_score" in available:
        available = available.sort_values(
            "selection_score",
            ascending=ascending,
        )

    for _, row in available.head(
        count
    ).iterrows():

        qid = str(row["qid"])
        used_qids.add(qid)

        selected_examples.append({
            "category": category,
            "qid": qid,
            "question":
                question_metadata[qid][
                    "question"
                ],
            "gold_answer_labels":
                question_metadata[qid][
                    "gold_answer_labels"
                ],
            "gold_answer_count": int(
                row["gold_answer_count"]
            ),
            "input_graph_edges": int(
                question_metadata[qid][
                    "input_graph_edges"
                ]
            ),
            "setstop_predicted_count": int(
                row[
                    "predicted_answer_count_setstop"
                ]
            ),
            "setstop_tp": int(
                row[
                    "true_positive_count_setstop"
                ]
            ),
            "setstop_fp": int(
                row[
                    "false_positive_count_setstop"
                ]
            ),
            "setstop_fn": int(
                row[
                    "false_negative_count_setstop"
                ]
            ),
            "setstop_f1": float(
                row["answer_f1_setstop"]
            ),
            "setstop_actions": int(
                row["actions_setstop"]
            ),
            "setstop_edges": int(
                row["retrieved_edges_setstop"]
            ),
            "setstop_stop_reason":
                row["stop_reason"],
            "setstop_premature": int(
                row["premature_stop"]
            ),
            "learned6_predicted_count": int(
                row[
                    "predicted_answer_count_learned6"
                ]
            ),
            "learned6_tp": int(
                row[
                    "true_positive_count_learned6"
                ]
            ),
            "learned6_fp": int(
                row[
                    "false_positive_count_learned6"
                ]
            ),
            "learned6_fn": int(
                row[
                    "false_negative_count_learned6"
                ]
            ),
            "learned6_f1": float(
                row["answer_f1_learned6"]
            ),
            "learned6_actions": int(
                row["actions_learned6"]
            ),
            "learned6_edges": int(
                row["retrieved_edges_learned6"]
            ),
        })


success_candidates = diagnostic_df[
    (
        diagnostic_df[
            "exact_match_setstop"
        ] == 1
    )
    & (
        diagnostic_df[
            "exact_match_learned6"
        ] == 0
    )
].copy()

success_candidates[
    "selection_score"
] = success_candidates[
    "f1_improvement"
]

select_examples(
    category=(
        "SetStop exact; fixed cap-6 "
        "ranker incorrect"
    ),
    candidates=success_candidates,
    count=3,
)


efficient_candidates = diagnostic_df[
    (
        diagnostic_df[
            "answer_f1_setstop"
        ]
        >= diagnostic_df[
            "answer_f1_learned6"
        ]
    )
    & (
        diagnostic_df[
            "actions_setstop"
        ]
        < diagnostic_df[
            "actions_learned6"
        ]
    )
].copy()

efficient_candidates[
    "selection_score"
] = (
    efficient_candidates[
        "f1_improvement"
    ]
    + 0.05
    * (
        efficient_candidates[
            "actions_learned6"
        ]
        - efficient_candidates[
            "actions_setstop"
        ]
    )
)

select_examples(
    category="Efficient SetStop success",
    candidates=efficient_candidates,
    count=2,
)


premature_candidates = diagnostic_df[
    (
        diagnostic_df[
            "premature_stop"
        ] == 1
    )
    & (
        diagnostic_df[
            "false_negative_count_setstop"
        ] > 0
    )
].copy()

premature_candidates[
    "selection_score"
] = premature_candidates[
    "false_negative_count_setstop"
]

select_examples(
    category="Premature-stop failure",
    candidates=premature_candidates,
    count=3,
)


overprediction_candidates = diagnostic_df[
    (
        diagnostic_df[
            "false_positive_count_setstop"
        ] > 0
    )
    & (
        diagnostic_df[
            "false_negative_count_setstop"
        ] == 0
    )
].copy()

overprediction_candidates[
    "selection_score"
] = overprediction_candidates[
    "false_positive_count_setstop"
]

select_examples(
    category="Over-prediction failure",
    candidates=overprediction_candidates,
    count=2,
)


empty_candidates = diagnostic_df[
    diagnostic_df[
        "predicted_answer_count_setstop"
    ] == 0
].copy()

empty_candidates[
    "selection_score"
] = empty_candidates[
    "gold_answer_count"
]

select_examples(
    category="Empty-prediction failure",
    candidates=empty_candidates,
    count=2,
)


diagnostic_examples_json = (
    OUTPUT_ROOT
    / "diagnostic_examples_v1.json"
)

diagnostic_examples_csv = (
    OUTPUT_ROOT
    / "diagnostic_examples_v1.csv"
)

write_json(
    diagnostic_examples_json,
    selected_examples,
)

pd.DataFrame(
    selected_examples
).to_csv(
    diagnostic_examples_csv,
    index=False,
)


# =============================================================================
# 11. CREATE THE FINAL TWO-PANEL PAPER FIGURE
# =============================================================================

import matplotlib

matplotlib.use(
    "Agg",
    force=True,
)

import matplotlib.pyplot as plt

try:
    plt.switch_backend("Agg")
except Exception:
    pass


COLORS = {
    "relation_fixed":
        "#8F969D",
    "learned_ranker_fixed":
        "#3478E5",
    "setstop":
        "#E94335",
}

MARKERS = {
    "relation_fixed":
        "o",
    "learned_ranker_fixed":
        "s",
    "setstop":
        "*",
}

LABELS = {
    "relation_fixed":
        "Relation fixed",
    "learned_ranker_fixed":
        "Learned ranker",
    "setstop":
        "SetStop",
}


figure, axes = plt.subplots(
    1,
    2,
    figsize=(12.6, 4.8),
    sharex=True,
)

panel_settings = [
    (
        axes[0],
        "macro_f1",
        "macro_f1_ci95_lower",
        "macro_f1_ci95_upper",
        "Macro answer-set F1",
        "(a) Answer-set quality",
    ),
    (
        axes[1],
        "complete_recall",
        "complete_recall_ci95_lower",
        "complete_recall_ci95_upper",
        "Complete-answer recall",
        "(b) Gold-set coverage",
    ),
]


for (
    axis,
    metric,
    lower_key,
    upper_key,
    ylabel,
    title,
) in panel_settings:

    for policy in [
        "relation_fixed",
        "learned_ranker_fixed",
        "setstop",
    ]:
        rows = budget_df[
            budget_df["policy"] == policy
        ].sort_values("budget_cap")

        x_values = rows[
            "budget_cap"
        ].to_numpy(dtype=float)

        y_values = rows[
            metric
        ].to_numpy(dtype=float)

        lower_values = rows[
            lower_key
        ].to_numpy(dtype=float)

        upper_values = rows[
            upper_key
        ].to_numpy(dtype=float)

        axis.plot(
            x_values,
            y_values,
            marker=MARKERS[policy],
            markersize=(
                10
                if policy == "setstop"
                else 5.5
            ),
            linewidth=(
                2.5
                if policy == "setstop"
                else 2.0
            ),
            color=COLORS[policy],
            label=LABELS[policy],
            zorder=3,
        )

        axis.fill_between(
            x_values,
            lower_values,
            upper_values,
            color=COLORS[policy],
            alpha=0.11,
            linewidth=0,
            zorder=1,
        )

    axis.set_xticks(
        [2, 4, 6, 8, 10, 12]
    )

    axis.set_xlabel(
        "Maximum action budget"
    )

    axis.set_ylabel(ylabel)

    axis.set_title(
        title,
        fontweight="bold",
    )

    axis.grid(
        alpha=0.22,
        linewidth=0.8,
    )

    axis.spines[
        "top"
    ].set_visible(False)

    axis.spines[
        "right"
    ].set_visible(False)


handles, labels = (
    axes[0].get_legend_handles_labels()
)

figure.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.035),
    ncol=3,
    frameon=False,
)

figure.tight_layout(
    rect=[0, 0, 1, 0.92]
)

combined_png = (
    OUTPUT_ROOT
    / "combined_budget_figure_v1.png"
)

combined_pdf = (
    OUTPUT_ROOT
    / "combined_budget_figure_v1.pdf"
)

figure.savefig(
    combined_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.close(figure)

png_to_pdf(
    combined_png,
    combined_pdf,
)


figure_caption_path = (
    OUTPUT_ROOT
    / "combined_budget_figure_caption_v1.tex"
)

figure_caption_path.write_text(
    (
        r"\caption{Performance across predefined "
        r"action budgets on the 736-question WebQSP "
        r"multi-answer subset. Shaded regions are "
        r"pointwise bootstrap 95\% confidence intervals. "
        r"SetStop improves answer-set F1 from cap 4 "
        r"onward, while fixed-budget ranking attains "
        r"higher raw complete-answer recall at larger "
        r"caps by continuing to retrieve more entities. "
        r"Significance claims in the text use paired "
        r"bootstrap confidence intervals.}"
        "\n"
    ),
    encoding="utf-8",
)


# =============================================================================
# 12. PAPER-SAFE CLAIMS AND RESULTS PARAGRAPH
# =============================================================================

primary_comparison = (
    cost_comparisons[
        "test"
    ][
        "setstop_vs_learned_cap6"
    ]
)

full_budget_pair = (
    cost_paired[
        "test"
    ][
        "same_cap"
    ][
        "12"
    ][
        "versus_learned"
    ]
)

setstop_row = get_budget_row(
    "setstop",
    12,
)

learned6_row = get_budget_row(
    "learned_ranker_fixed",
    6,
)

learned12_row = get_budget_row(
    "learned_ranker_fixed",
    12,
)

learned_stops = int(
    (
        setstop_df["stop_reason"]
        == "learned_stop"
    ).sum()
)

premature_stops = int(
    setstop_df[
        "premature_stop"
    ].sum()
)

premature_among_learned = (
    premature_stops / learned_stops
    if learned_stops
    else 0.0
)


paper_claims = {
    "recommended_title": (
        "SetStop-KGQA: Adaptive Stopping "
        "for Cost-Aware Multi-Answer "
        "Knowledge Graph Question Answering"
    ),
    "primary_claim": (
        "At matched average action cost, "
        "SetStop improves macro answer-set F1 "
        "by 8.26 points and exact match by "
        "14.54 points over the learned "
        "fixed-budget ranker, while "
        "complete-answer recall and exposed-edge "
        "differences are not statistically "
        "significant."
    ),
    "primary_evidence": {
        "setstop_macro_f1": float(
            setstop_row["macro_f1"]
        ),
        "learned_cap6_macro_f1": float(
            learned6_row["macro_f1"]
        ),
        "macro_f1_difference":
            delta_f1[
                "mean_difference_first_minus_second"
            ],
        "macro_f1_ci95": [
            delta_f1["ci95_lower"],
            delta_f1["ci95_upper"],
        ],
        "exact_match_difference":
            delta_exact[
                "mean_difference_first_minus_second"
            ],
        "exact_match_ci95": [
            delta_exact["ci95_lower"],
            delta_exact["ci95_upper"],
        ],
        "complete_recall_difference":
            delta_complete[
                "mean_difference_first_minus_second"
            ],
        "complete_recall_ci95": [
            delta_complete["ci95_lower"],
            delta_complete["ci95_upper"],
        ],
        "action_difference":
            delta_actions[
                "mean_difference_first_minus_second"
            ],
        "action_ci95": [
            delta_actions["ci95_lower"],
            delta_actions["ci95_upper"],
        ],
        "edge_difference":
            delta_edges[
                "mean_difference_first_minus_second"
            ],
        "edge_ci95": [
            delta_edges["ci95_lower"],
            delta_edges["ci95_upper"],
        ],
    },
    "secondary_claim": (
        "At the common maximum cap of 12, "
        "SetStop improves macro F1 by "
        f"{full_budget_pair['metrics']['answer_f1']['mean_difference_first_minus_second']:.4f} "
        "while reducing mean actions by 50.4% "
        "and input edges by 35.5%; however, "
        "complete-answer recall decreases by "
        "6.66 points."
    ),
    "external_baseline_claim": (
        "NSM+h and ReaRev achieve substantially "
        "higher answer accuracy but consume the "
        "entire question-specific input graph. "
        "SetStop exposes approximately 97.6% "
        "fewer input triples and should therefore "
        "be presented as an accuracy–graph-access "
        "trade-off, not as a state-of-the-art "
        "accuracy model."
    ),
    "stopping_statistics": {
        "learned_stop_questions":
            learned_stops,
        "learned_stop_fraction":
            learned_stops / 736,
        "premature_stop_questions":
            premature_stops,
        "premature_stop_fraction_all_questions":
            premature_stops / 736,
        "premature_fraction_among_learned_stops":
            premature_among_learned,
    },
    "paper_safe_wording": [
        (
            "adaptive stopping improves answer-set "
            "quality at matched average search cost"
        ),
        (
            "SetStop dynamically reallocates "
            "search effort across questions"
        ),
        (
            "SetStop provides a favorable "
            "accuracy–graph-access operating point"
        ),
        (
            "complete recall is maintained at "
            "matched action cost but not at the "
            "same maximum cap"
        ),
    ],
    "claims_to_avoid": [
        (
            "SetStop is state of the art "
            "on WebQSP accuracy"
        ),
        (
            "SetStop always finds a more "
            "complete answer set"
        ),
        (
            "input-edge reduction is equivalent "
            "to an equal FLOP or latency reduction"
        ),
        (
            "the method guarantees that all "
            "answers have been found"
        ),
    ],
    "limitations": [
        (
            "The controlled evaluation currently "
            "covers one dataset."
        ),
        (
            "Approximately half of learned stopping "
            "events are premature according to "
            "oracle reachability."
        ),
        (
            "The learned answer recognizer remains "
            "a major source of empty and mixed-error "
            "predictions."
        ),
        (
            "Input triples exposed is a graph-access "
            "proxy, not a direct measurement of "
            "FLOPs, memory, energy, or wall-clock "
            "latency."
        ),
        (
            "Full-graph message-passing baselines "
            "and sequential frontier search use "
            "different computational structures."
        ),
    ],
}


paper_claims_path = (
    OUTPUT_ROOT
    / "paper_claims_v1.json"
)

write_json(
    paper_claims_path,
    paper_claims,
)


results_paragraph = (
    r"\paragraph{Cost-matched evaluation.} "
    f"At approximately the same average action "
    f"cost, SetStop obtains a macro answer-set F1 "
    f"of {float(setstop_row['macro_f1']):.3f}, "
    f"compared with "
    f"{float(learned6_row['macro_f1']):.3f} "
    f"for the learned fixed-budget ranker at six "
    f"actions. This is an absolute improvement of "
    f"{delta_f1['mean_difference_first_minus_second']:.3f} "
    f"(paired 95\\% CI "
    f"[{delta_f1['ci95_lower']:.3f}, "
    f"{delta_f1['ci95_upper']:.3f}]). "
    f"Exact match improves from "
    f"{float(learned6_row['exact_match']):.3f} "
    f"to {float(setstop_row['exact_match']):.3f}. "
    f"The methods use "
    f"{float(learned6_row['mean_actions']):.2f} "
    f"and "
    f"{float(setstop_row['mean_actions']):.2f} "
    f"actions on average, respectively, and their "
    f"exposed-edge difference is not significant. "
    f"Complete-answer recall is also statistically "
    f"indistinguishable at matched cost. At the "
    f"common maximum cap of 12, SetStop reduces "
    f"actions by "
    f"{operating_summary['setstop_vs_fixed_ranker']['action_reduction_percent']:.1f}\\% "
    f"and input edges by "
    f"{operating_summary['setstop_vs_fixed_ranker']['edge_reduction_percent']:.1f}\\%, "
    f"but sacrifices "
    f"{abs(float(setstop_row['complete_recall']) - float(learned12_row['complete_recall'])):.3f} "
    f"complete-recall points."
)

results_paragraph_path = (
    OUTPUT_ROOT
    / "results_paragraph_v1.tex"
)

results_paragraph_path.write_text(
    results_paragraph + "\n",
    encoding="utf-8",
)


# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================

summary_output = {
    "notebook": (
        "10_SetStop_Final_Paper_Analysis_"
        "and_LaTeX_Tables.ipynb"
    ),
    "runtime": "CPU",
    "model_inference_performed": False,
    "test_questions": 736,
    "qid_sha256": qid_digest,
    "notebook06_regression_check":
        cost_summary[
            "regression_check"
        ],
    "primary_cost_matched_result":
        primary_comparison,
    "paired_diagnostics":
        paired_diagnostics,
    "setstop_error_categories":
        error_df.to_dict(
            orient="records"
        ),
    "setstop_stop_outcomes":
        stop_df.to_dict(
            orient="records"
        ),
    "setstop_answer_count_results":
        answer_df.to_dict(
            orient="records"
        ),
    "external_graph_exposure": {
        "setstop_mean_edges":
            setstop_edges,
        "full_graph_baseline_mean_edges":
            full_graph_edges,
        "setstop_edge_reduction_percent":
            external_paired[
                "comparisons"
            ][
                "NSM+h"
            ][
                "mean_edge_reduction_percent"
            ],
    },
    "paper_claims":
        paper_claims,
    "runtime_seconds": float(
        time.time() - START_TIME
    ),
}

summary_path = (
    OUTPUT_ROOT
    / "final_paper_analysis_summary_v1.json"
)

write_json(
    summary_path,
    summary_output,
)


# =============================================================================
# 14. ARTIFACT INVENTORY
# =============================================================================

artifact_paths = [
    main_results_csv,
    main_results_tex,
    cost_table_csv,
    cost_table_tex,
    budget_table_csv,
    budget_table_tex,
    error_csv,
    error_tex,
    stop_csv,
    stop_tex,
    answer_csv,
    answer_tex,
    paired_diagnostics_path,
    diagnostic_examples_json,
    diagnostic_examples_csv,
    combined_png,
    combined_pdf,
    figure_caption_path,
    paper_claims_path,
    results_paragraph_path,
    summary_path,
]

for path in artifact_paths:
    assert (
        path.exists()
        and path.stat().st_size > 0
    ), f"Missing output artifact: {path}"


inventory = {
    "artifacts": [
        {
            "name": path.name,
            "path": str(path),
            "bytes": int(
                path.stat().st_size
            ),
            "sha256": sha256_file(path),
        }
        for path in artifact_paths
    ]
}

inventory_path = (
    OUTPUT_ROOT
    / "artifact_inventory_v1.json"
)

write_json(
    inventory_path,
    inventory,
)


# =============================================================================
# 15. FINAL CONSOLE REPORT
# =============================================================================

print("\n" + "=" * 110)
print("MAIN PAPER RESULTS")
print("=" * 110)

print(
    main_results_df[
        [
            "group",
            "method",
            "macro_f1",
            "exact_match",
            "complete_recall",
            "mean_actions",
            "mean_input_edges",
        ]
    ].to_string(
        index=False,
        float_format=(
            lambda value: f"{value:.4f}"
        ),
    )
)


print("\n" + "=" * 110)
print("COST-MATCHED RESULTS")
print("=" * 110)

print(
    cost_table_df.to_string(
        index=False,
        float_format=(
            lambda value: f"{value:.4f}"
        ),
    )
)

print(
    "\nSetStop minus learned cap-6 F1: "
    f"{delta_f1['mean_difference_first_minus_second']:+.4f} "
    f"[{delta_f1['ci95_lower']:+.4f}, "
    f"{delta_f1['ci95_upper']:+.4f}]"
)

print(
    "Exact-match McNemar p-value:",
    f"{paired_diagnostics['exact_match_mcnemar']['exact_two_sided_p_value']:.6g}",
)

print(
    "Complete-recall McNemar p-value:",
    f"{paired_diagnostics['complete_recall_mcnemar']['exact_two_sided_p_value']:.6g}",
)


print("\n" + "=" * 110)
print("SETSTOP ERROR BREAKDOWN")
print("=" * 110)

print(
    error_df[
        [
            "error_category",
            "questions",
            "percentage",
            "macro_f1",
            "mean_actions",
        ]
    ].to_string(
        index=False,
        float_format=(
            lambda value: f"{value:.4f}"
        ),
    )
)


print("\n" + "=" * 110)
print("STOP OUTCOME BREAKDOWN")
print("=" * 110)

print(
    stop_df[
        [
            "stop_outcome",
            "questions",
            "percentage",
            "macro_f1",
            "exact_match",
            "complete_recall",
            "mean_actions",
        ]
    ].to_string(
        index=False,
        float_format=(
            lambda value: f"{value:.4f}"
        ),
    )
)


print(
    "\nF1 win/tie/loss against "
    "the learned cap-6 ranker:"
)

print(
    paired_diagnostics[
        "f1_win_tie_loss"
    ]
)


print("\nGenerated paper artifacts:")

for path in (
    artifact_paths
    + [inventory_path]
):
    print(path)


print(
    "\nSend these files for "
    "final paper drafting:"
)

print(summary_path)
print(main_results_tex)
print(cost_table_tex)
print(budget_table_tex)
print(error_tex)
print(combined_png)
print(paper_claims_path)
print(results_paragraph_path)

print("=" * 110)
print(
    "FINAL PAPER ANALYSIS "
    "COMPLETED SUCCESSFULLY"
)
print("=" * 110)

Mounted at /content/drive
SETSTOP-KGQA FINAL PAPER ANALYSIS AND LATEX TABLES
Runtime: CPU — no GPU or model inference is required
Python: 3.13.15
Pandas: 2.2.3
Output directory: /content/drive/MyDrive/SetStop_KGQA/outputs/final_paper_analysis_v1
Controlled test subset verified: 736 questions
QID SHA-256: a1881ab84e5ccafd18ae99f11df831202ec447b155056ea56cf16f9c9178ef85
Notebook-06 cap-12 reproduction: PASS

MAIN PAPER RESULTS
              group                  method  macro_f1  exact_match  complete_recall  mean_actions  mean_input_edges
  Sequential search Relation fixed (cap 12)    0.3017       0.1128           0.4062       12.0000          148.5557
  Sequential search Learned ranker (cap 12)    0.3384       0.1277           0.4592       12.0000          171.0367
  Sequential search                 SetStop    0.3856       0.2772           0.3927        5.9524          110.2758
  Sequential search           Oracle ranker    0.4698       0.3383           0.4755        8.0978          